In [0]:
%sql
-- Customer Age Distribution: Count customers by exact age

SELECT
    current_age,
    COUNT(*) AS customer_count
FROM adbdevbankproject.gold.dim_users
GROUP BY current_age
ORDER BY current_age;

current_age,customer_count
18,77
19,34
20,40
21,40
22,43
23,32
24,37
25,34
26,39
27,26


In [0]:
%sql
-- Customer Age Distribution: Analyze current age across all customers

SELECT
    MIN(current_age) AS min_age,
    MAX(current_age) AS max_age,
    ROUND(AVG(current_age), 2) AS avg_age,
    COUNT(*) AS total_customers
FROM adbdevbankproject.gold.dim_users;

min_age,max_age,avg_age,total_customers
18,101,45.39,2000


### # Gold Layer is complete and modeled as a Star Schema with surrogate keys, primary keys, foreign keys, and validated referential integrity.

In [0]:
%sql
-- 6. Validate Business Keys are preserved

SELECT
    'users' AS dimension,
    COUNT(*) AS mismatched_records
FROM adbdevbankproject.gold.dim_users g
LEFT JOIN adbdevbankproject.silver.users_data s
    ON g.client_id = s.id
WHERE s.id IS NULL

UNION ALL

SELECT
    'cards',
    COUNT(*)
FROM adbdevbankproject.gold.dim_cards g
LEFT JOIN adbdevbankproject.silver.cards_data s
    ON g.card_id = s.id
WHERE s.id IS NULL

UNION ALL

SELECT
    'mcc',
    COUNT(*)
FROM adbdevbankproject.gold.dim_mcc g
LEFT JOIN adbdevbankproject.silver.mcc_codes s
    ON g.mcc_code = s.mcc_code
WHERE s.mcc_code IS NULL;

dimension,mismatched_records
users,0
cards,0
mcc,0


In [0]:
%sql
-- 5. Verify every Silver transaction exists in Gold

SELECT
    COUNT(*) AS missing_transactions
FROM adbdevbankproject.silver.transactions_data s
LEFT JOIN adbdevbankproject.gold.fact_transactions f
    ON s.id = f.transaction_id
WHERE f.transaction_id IS NULL;

missing_transactions
0


In [0]:
%sql
-- 4. Check unmatched Foreign Keys

SELECT
    COUNT(*) AS total_transactions,

    SUM(CASE WHEN u.user_key IS NULL THEN 1 ELSE 0 END) AS missing_users,

    SUM(CASE WHEN c.card_key IS NULL THEN 1 ELSE 0 END) AS missing_cards,

    SUM(CASE WHEN m.mcc_key IS NULL THEN 1 ELSE 0 END) AS missing_mcc,

    SUM(CASE WHEN d.date_key IS NULL THEN 1 ELSE 0 END) AS missing_dates

FROM adbdevbankproject.gold.fact_transactions f

LEFT JOIN adbdevbankproject.gold.dim_users u
    ON f.user_key = u.user_key

LEFT JOIN adbdevbankproject.gold.dim_cards c
    ON f.card_key = c.card_key

LEFT JOIN adbdevbankproject.gold.dim_mcc m
    ON f.mcc_key = m.mcc_key

LEFT JOIN adbdevbankproject.gold.dim_date d
    ON f.date_key = d.date_key;

total_transactions,missing_users,missing_cards,missing_mcc,missing_dates
13305909,0,0,0,0


In [0]:
%sql
-- 3. Check duplicate Dimension Primary Keys

SELECT 'dim_users' AS table_name, COUNT(*) AS duplicate_keys
FROM (
    SELECT user_key
    FROM adbdevbankproject.gold.dim_users
    GROUP BY user_key
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT 'dim_cards', COUNT(*)
FROM (
    SELECT card_key
    FROM adbdevbankproject.gold.dim_cards
    GROUP BY card_key
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT 'dim_mcc', COUNT(*)
FROM (
    SELECT mcc_key
    FROM adbdevbankproject.gold.dim_mcc
    GROUP BY mcc_key
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT 'dim_date', COUNT(*)
FROM (
    SELECT date_key
    FROM adbdevbankproject.gold.dim_date
    GROUP BY date_key
    HAVING COUNT(*) > 1
);

table_name,duplicate_keys
dim_users,0
dim_cards,0
dim_mcc,0
dim_date,0


In [0]:
%sql
-- 2. Check duplicate transaction IDs

SELECT
    transaction_id,
    COUNT(*) AS record_count
FROM adbdevbankproject.gold.fact_transactions
GROUP BY transaction_id
HAVING COUNT(*) > 1
LIMIT 20;

transaction_id,record_count


In [0]:
%sql
-- 1. Compare Silver vs Gold transaction counts

SELECT
    (SELECT COUNT(*)
     FROM adbdevbankproject.silver.transactions_data) AS silver_count,

    (SELECT COUNT(*)
     FROM adbdevbankproject.gold.fact_transactions) AS gold_count;

silver_count,gold_count
13305909,13305909


In [0]:
%sql
-- Foreign Key: fact_transactions → dim_date

ALTER TABLE adbdevbankproject.gold.fact_transactions
ADD CONSTRAINT fk_fact_date
FOREIGN KEY (date_key)
REFERENCES adbdevbankproject.gold.dim_date (date_key);

In [0]:
%sql
-- Foreign Key: fact_transactions → dim_mcc

ALTER TABLE adbdevbankproject.gold.fact_transactions
ADD CONSTRAINT fk_fact_mcc
FOREIGN KEY (mcc_key)
REFERENCES adbdevbankproject.gold.dim_mcc (mcc_key);

In [0]:
%sql
-- Foreign Key: fact_transactions → dim_cards

ALTER TABLE adbdevbankproject.gold.fact_transactions
ADD CONSTRAINT fk_fact_cards
FOREIGN KEY (card_key)
REFERENCES adbdevbankproject.gold.dim_cards (card_key);

In [0]:
%sql
-- Foreign Key: fact_transactions → dim_users

ALTER TABLE adbdevbankproject.gold.fact_transactions
ADD CONSTRAINT fk_fact_users
FOREIGN KEY (user_key)
REFERENCES adbdevbankproject.gold.dim_users (user_key);

In [0]:
%sql
-- Check for unmatched Dimension keys

SELECT
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN user_key IS NULL THEN 1 ELSE 0 END) AS missing_user_key,
    SUM(CASE WHEN card_key IS NULL THEN 1 ELSE 0 END) AS missing_card_key,
    SUM(CASE WHEN mcc_key IS NULL THEN 1 ELSE 0 END) AS missing_mcc_key,
    SUM(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END) AS missing_date_key
FROM adbdevbankproject.gold.fact_transactions;

total_transactions,missing_user_key,missing_card_key,missing_mcc_key,missing_date_key
13305909,0,0,0,0


In [0]:
%sql

SELECT DISTINCT errors
FROM adbdevbankproject.gold.fact_transactions

errors
"Bad Zipcode,Technical Glitch"
Bad Expiration
"Bad PIN,Technical Glitch"
"Bad PIN,Insufficient Balance"
"Bad Zipcode,Insufficient Balance"
Insufficient Balance
Bad Card Number
"Insufficient Balance,Technical Glitch"
"Bad Card Number,Insufficient Balance"
"Bad CVV,Technical Glitch"


In [0]:
%sql
-- Preview Gold Fact

SELECT *
FROM adbdevbankproject.gold.fact_transactions
LIMIT 10;

transaction_id,user_key,card_key,mcc_key,date_key,transaction_date,transaction_time,amount,use_chip,merchant_id,merchant_city,merchant_state,merchant_country,zip,errors
16571701,1718,2653,44,20150825,2015-08-25,13:54:00,60.00,Chip Transaction,27092,Buffalo,NY,United States,14219,No Error
16571702,1738,1097,57,20150825,2015-08-25,13:54:00,10.43,Chip Transaction,27313,Eastpointe,MI,United States,48021,No Error
16571703,134,2199,45,20150825,2015-08-25,13:55:00,164.61,Online Transaction,4802,null,null,null,null,No Error
16571706,491,177,71,20150825,2015-08-25,13:55:00,21.96,Chip Transaction,49194,Hagerman,NM,United States,88232,No Error
16571707,821,128,56,20150825,2015-08-25,13:55:00,43.44,Online Transaction,88459,null,null,null,null,No Error
16571708,901,3413,79,20150825,2015-08-25,13:55:00,14.26,Chip Transaction,20519,Pensacola,FL,United States,32501,No Error
16571709,1042,2501,91,20150825,2015-08-25,13:55:00,77.47,Swipe Transaction,13646,Evans,GA,United States,30809,No Error
16571711,1124,4351,60,20150825,2015-08-25,13:55:00,21.58,Chip Transaction,61195,Laurel Hill,NC,United States,28351,No Error
16571710,1124,4351,60,20150825,2015-08-25,13:55:00,-96.00,Chip Transaction,61195,Laurel Hill,NC,United States,28351,No Error
16571712,209,4981,36,20150825,2015-08-25,13:56:00,37.21,Online Transaction,18563,null,null,null,null,No Error


In [0]:
%sql
-- Load transactions and resolve Dimension Surrogate Keys

INSERT INTO adbdevbankproject.gold.fact_transactions
SELECT
    t.id AS transaction_id,

    u.user_key,
    c.card_key,
    m.mcc_key,
    d.date_key,

    t.transaction_date,
    t.transaction_time,
    t.amount,
    t.use_chip,
    t.merchant_id,
    t.merchant_city,
    t.merchant_state,
    t.merchant_country,
    t.zip,
    t.errors

FROM adbdevbankproject.silver.transactions_data t

LEFT JOIN adbdevbankproject.gold.dim_users u
    ON t.client_id = u.client_id

LEFT JOIN adbdevbankproject.gold.dim_cards c
    ON t.card_id = c.card_id

LEFT JOIN adbdevbankproject.gold.dim_mcc m
    ON t.mcc = m.mcc_code

LEFT JOIN adbdevbankproject.gold.dim_date d
    ON t.transaction_date = d.full_date;

num_affected_rows,num_inserted_rows
13305909,13305909


In [0]:
%sql
-- Gold Fact: Transactions
-- Grain: One row per transaction

CREATE TABLE adbdevbankproject.gold.fact_transactions (
    transaction_id INT NOT NULL,

    user_key BIGINT NOT NULL,
    card_key BIGINT NOT NULL,
    mcc_key BIGINT NOT NULL,
    date_key INT NOT NULL,

    transaction_date DATE,
    transaction_time STRING,
    amount DECIMAL(18,2),
    use_chip STRING,
    merchant_id INT,
    merchant_city STRING,
    merchant_state STRING,
    merchant_country STRING,
    zip STRING,
    errors STRING,

    CONSTRAINT pk_fact_transactions
        PRIMARY KEY (transaction_id)
)
USING DELTA;

In [0]:
%sql
-- Preview Date Dimension

SELECT *
FROM adbdevbankproject.gold.dim_date
ORDER BY full_date
LIMIT 10;

date_key,full_date,day_of_month,day_name,day_of_week,week_of_year,month_number,month_name,quarter_number,year_number
20100101,2010-01-01,1,Friday,6,53,1,January,1,2010
20100102,2010-01-02,2,Saturday,7,53,1,January,1,2010
20100103,2010-01-03,3,Sunday,1,53,1,January,1,2010
20100104,2010-01-04,4,Monday,2,1,1,January,1,2010
20100105,2010-01-05,5,Tuesday,3,1,1,January,1,2010
20100106,2010-01-06,6,Wednesday,4,1,1,January,1,2010
20100107,2010-01-07,7,Thursday,5,1,1,January,1,2010
20100108,2010-01-08,8,Friday,6,1,1,January,1,2010
20100109,2010-01-09,9,Saturday,7,1,1,January,1,2010
20100110,2010-01-10,10,Sunday,1,1,1,January,1,2010


In [0]:
%sql
-- Generate continuous calendar dates
-- From 2010-01-01 to 2019-10-31

INSERT INTO adbdevbankproject.gold.dim_date
SELECT
    CAST(date_format(date_value, 'yyyyMMdd') AS INT) AS date_key,
    date_value AS full_date,
    day(date_value) AS day_of_month,
    date_format(date_value, 'EEEE') AS day_name,
    dayofweek(date_value) AS day_of_week,
    weekofyear(date_value) AS week_of_year,
    month(date_value) AS month_number,
    date_format(date_value, 'MMMM') AS month_name,
    quarter(date_value) AS quarter_number,
    year(date_value) AS year_number
FROM (
    SELECT
        explode(
            sequence(
                DATE '2010-01-01',
                DATE '2019-10-31',
                INTERVAL 1 DAY
            )
        ) AS date_value
);

num_affected_rows,num_inserted_rows
3591,3591


In [0]:
%sql
-- Gold Dimension: Date
-- date_key = Surrogate/Date Key
-- One row per calendar date

CREATE TABLE adbdevbankproject.gold.dim_date (
    date_key INT NOT NULL,
    full_date DATE NOT NULL,
    day_of_month INT,
    day_name STRING,
    day_of_week INT,
    week_of_year INT,
    month_number INT,
    month_name STRING,
    quarter_number INT,
    year_number INT,

    CONSTRAINT pk_dim_date
        PRIMARY KEY (date_key)
)
USING DELTA;

In [0]:
%sql
-- Check transaction date range for Date Dimension

SELECT
    MIN(transaction_date) AS min_date,
    MAX(transaction_date) AS max_date
FROM adbdevbankproject.silver.transactions_data;

min_date,max_date
2010-01-01,2019-10-31


In [0]:
%sql
-- Preview Gold dim_mcc

SELECT *
FROM adbdevbankproject.gold.dim_mcc
ORDER BY mcc_key
LIMIT 10;

mcc_key,mcc_code,merchant_category
1,1711,"Heating, Plumbing, Air Conditioning Contractors"
2,3000,Steelworks
3,3001,Steel Products Manufacturing
4,3005,Miscellaneous Metal Fabrication
5,3006,Miscellaneous Fabricated Metal Products
6,3007,Coated and Laminated Products
7,3008,Steel Drums and Barrels
8,3009,Fabricated Structural Metal Products
9,3058,"Tools, Parts, Supplies Manufacturing"
10,3066,Miscellaneous Metals


In [0]:
%sql
-- Load Silver MCC data into Gold
-- Generate Surrogate Key for each MCC

INSERT INTO adbdevbankproject.gold.dim_mcc
SELECT
    ROW_NUMBER() OVER (ORDER BY mcc_code) AS mcc_key,
    mcc_code,
    merchant_category
FROM adbdevbankproject.silver.mcc_codes;

num_affected_rows,num_inserted_rows
109,109


In [0]:
%sql
-- Gold Dimension: Merchant Category
-- mcc_key  = Surrogate Primary Key
-- mcc_code = Original Business Key from Silver

CREATE TABLE adbdevbankproject.gold.dim_mcc (
    mcc_key BIGINT NOT NULL,
    mcc_code INT NOT NULL,
    merchant_category STRING,

    CONSTRAINT pk_dim_mcc
        PRIMARY KEY (mcc_key)
)
USING DELTA;

In [0]:
%sql
-- Preview Gold dim_cards

SELECT *
FROM adbdevbankproject.gold.dim_cards
ORDER BY card_key
LIMIT 10;

card_key,card_id,client_id,card_brand,card_type,expires,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
1,0,1362,Amex,Credit,2024-04-01,true,2,33900.00,1991-01-01,2014,false
2,1,550,Mastercard,Credit,2024-06-01,true,1,11600.00,1994-01-01,2013,false
3,2,556,Mastercard,Debit,2021-09-01,true,1,19948.00,1995-01-01,2011,false
4,3,1937,Visa,Credit,2020-04-01,true,2,16400.00,1995-01-01,2015,false
5,4,1981,Mastercard,Debit,2024-03-01,true,2,19439.00,1997-01-01,2007,false
6,5,619,Visa,Debit,2024-04-01,true,2,21883.00,1997-01-01,2012,false
7,6,1046,Amex,Credit,1999-02-01,true,2,9400.00,1998-01-01,2011,false
8,7,511,Mastercard,Debit,2005-03-01,true,1,9664.00,1998-01-01,2011,false
9,8,1107,Mastercard,Credit,2021-09-01,false,2,10300.00,1998-01-01,2006,false
10,9,1046,Amex,Credit,2020-09-01,true,1,13000.00,1999-01-01,2005,false


In [0]:
%sql
-- Load Silver cards into Gold
-- Generate Surrogate Key for each card

INSERT INTO adbdevbankproject.gold.dim_cards
SELECT
    ROW_NUMBER() OVER (ORDER BY id) AS card_key,
    id AS card_id,
    client_id,
    card_brand,
    card_type,
    expires,
    has_chip,
    num_cards_issued,
    credit_limit,
    acct_open_date,
    year_pin_last_changed,
    card_on_dark_web
FROM adbdevbankproject.silver.cards_data;

num_affected_rows,num_inserted_rows
6146,6146


In [0]:
%sql
-- Gold Dimension: Cards
-- card_key = Surrogate Primary Key
-- card_id  = Original Business Key from Silver

CREATE TABLE adbdevbankproject.gold.dim_cards (
    card_key BIGINT NOT NULL,
    card_id INT NOT NULL,
    client_id INT NOT NULL,
    card_brand STRING,
    card_type STRING,
    expires DATE,
    has_chip BOOLEAN,
    num_cards_issued INT,
    credit_limit DECIMAL(12,2),
    acct_open_date DATE,
    year_pin_last_changed INT,
    card_on_dark_web BOOLEAN,

    CONSTRAINT pk_dim_cards
        PRIMARY KEY (card_key)
)
USING DELTA;

In [0]:
%sql
-- Check original client IDs in Silver

SELECT
    id AS client_id
FROM adbdevbankproject.silver.users_data
ORDER BY id

client_id
0
1
2
3
4
5
6
7
8
9


In [0]:
%sql
-- Preview Gold dim_users

SELECT *
FROM adbdevbankproject.gold.dim_users


user_key,client_id,current_age,retirement_age,birth_year,birth_month,gender,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
1,0,33,69,1986,3,Male,43.59,-70.33,29237.00,59613.00,36199.00,763,4
2,1,43,74,1976,4,Female,30.44,-87.18,22247.00,45360.00,14587.00,704,3
3,2,48,64,1971,8,Male,40.84,-73.87,13461.00,27447.00,80850.00,673,5
4,3,49,65,1970,12,Male,33.89,-98.51,13705.00,27943.00,18693.00,681,4
5,4,54,72,1965,3,Female,47.61,-122.3,37485.00,76431.00,115362.00,716,5
6,5,65,65,1955,2,Male,42.95,-77.13,19095.00,20614.00,14042.00,711,2
7,6,19,63,2000,6,Female,32.88,-117.13,27394.00,55854.00,111042.00,782,1
8,7,74,61,1945,5,Male,39.3,-76.61,20919.00,32682.00,21379.00,650,3
9,8,21,69,1998,8,Male,27.95,-82.48,18881.00,38497.00,33413.00,699,2
10,9,29,60,1990,12,Male,32.63,-117.05,20102.00,40988.00,89801.00,540,3


In [0]:
%sql
-- Load users and generate Surrogate Keys

INSERT INTO adbdevbankproject.gold.dim_users
SELECT
    ROW_NUMBER() OVER (ORDER BY id) AS user_key,
    id AS client_id,
    current_age,
    retirement_age,
    birth_year,
    birth_month,
    gender,
    latitude,
    longitude,
    per_capita_income,
    yearly_income,
    total_debt,
    credit_score,
    num_credit_cards
FROM adbdevbankproject.silver.users_data;

num_affected_rows,num_inserted_rows
2000,2000


In [0]:
%sql
-- Gold Dimension: Users
-- user_key = Surrogate Key
-- client_id = Business Key

CREATE TABLE adbdevbankproject.gold.dim_users (
    user_key BIGINT NOT NULL,
    client_id INT NOT NULL,
    current_age INT,
    retirement_age INT,
    birth_year INT,
    birth_month INT,
    gender STRING,
    latitude DOUBLE,
    longitude DOUBLE,
    per_capita_income DECIMAL(12,2),
    yearly_income DECIMAL(12,2),
    total_debt DECIMAL(12,2),
    credit_score INT,
    num_credit_cards INT,

    CONSTRAINT pk_dim_users
        PRIMARY KEY (user_key)
)
USING DELTA;



In [0]:
%sql
-- Verify Gold schema location

DESCRIBE SCHEMA EXTENDED adbdevbankproject.gold;

database_description_item,database_description_value
Catalog Name,adbdevbankproject
Namespace Name,bronze
Comment,
Location,
Owner,ohoud.hrb@gmail.com
Properties,
Predictive Optimization,ENABLE (inherited from METASTORE metastore_azure_uaenorth)


In [0]:
%sql
-- Create Gold schema with Gold storage location

CREATE SCHEMA adbdevbankproject.gold
MANAGED LOCATION 'abfss://gold@stgdevbankproject.dfs.core.windows.net/';;

In [0]:
%sql
-- Review Silver schemas for all four tables

DESCRIBE TABLE adbdevbankproject.silver.transactions_data;

DESCRIBE TABLE adbdevbankproject.silver.users_data;

DESCRIBE TABLE adbdevbankproject.silver.cards_data;

DESCRIBE TABLE adbdevbankproject.silver.mcc_codes;


col_name,data_type,comment
mcc_code,int,null
merchant_category,string,null
